In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import xgboost as xgb
import lightgbm as lgb

from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, MACCSkeys, AllChem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

import warnings
warnings.filterwarnings(action='ignore')
warnings.simplefilter('ignore')

In [ ]:
print("데이터 로딩 중...")
train = pd.read_csv("chembl_preprocessed(N_1671).csv")

test = pd.read_csv("test.csv")
submission = pd.read_csv("sample_submission.csv")

In [ ]:
# print("데이터 로딩 중...")
# train_base = pd.read_csv("train.csv")
# # train_plus = pd.read_csv("train_chembl.csv")
# train_plus = pd.read_csv("chembl_preprocessed(N_1671).csv")

# test = pd.read_csv("test.csv")
# submission = pd.read_csv("sample_submission.csv")

# temp_df1 = train_base.drop(['ID'],axis=1)
# # temp_df2 = train_plus.drop(['Unnamed: 0'],axis=1)

# temp_df2 = train_plus

In [ ]:
# # --- 1단계: Inhibition 구간 정의 및 분포 분석 ---
# print("--- 1. 처리 전 temp_df1의 분포 ---")
# # Inhibition을 5단위 구간으로 나누기 위한 bin 생성
# min_val = 0
# max_val = int(np.ceil(max(temp_df1['Inhibition'].max(), temp_df2['Inhibition'].max())))
# bins = range(min_val, max_val + 5, 5)
# labels = range(len(bins) - 1)

# # 각 DataFrame에 'inhibition_bin' 열 추가
# temp_df1['inhibition_bin'] = pd.cut(temp_df1['Inhibition'], bins=bins, right=False, labels=labels)
# temp_df2['inhibition_bin'] = pd.cut(temp_df2['Inhibition'], bins=bins, right=False, labels=labels)

# # temp_df1의 구간별 데이터 수 확인
# df1_bin_counts = temp_df1['inhibition_bin'].value_counts().sort_index()
# print(df1_bin_counts)

In [ ]:
# print("처리 전 temp_df2의 분포")
# df2_bin_counts = temp_df2['inhibition_bin'].value_counts().sort_index()
# print(df2_bin_counts)
# print("\n")

In [ ]:
# # --- 2단계: temp_df1의 부족한 구간을 temp_df2에서 보충 ---
# print("--- 2. temp_df1 보충 작업 ---")
# # 목표 수량은 temp_df1에서 가장 많은 구간의 데이터 수
# target_count = df1_bin_counts.max()
# print(f"보충 목표 수량 (temp_df1의 최대 구간 개수): {target_count}\n")

# data_to_add = [] # temp_df2에서 가져올 데이터를 담을 리스트

# # 모든 inhibition 구간에 대해 반복
# for bin_label in labels:
#     # 현재 구간에 있는 df1 데이터 수
#     current_count = df1_bin_counts.get(bin_label, 0)
    
#     # 보충이 필요한지 계산
#     num_to_add = target_count - current_count
    
#     if num_to_add > 0:
#         # df2에서 해당 구간의 데이터를 찾음
#         candidates = temp_df2[temp_df2['inhibition_bin'] == bin_label]
        
#         # df2에서 가져올 수 있는 실제 데이터 수
#         num_to_sample = min(num_to_add, len(candidates))
        
#         if num_to_sample > 0:
#             print(f"구간 {bin_label} ({bins[bin_label]}-{bins[bin_label+1]}): {current_count}개 -> {num_to_add}개 보충 필요 -> {num_to_sample}개 실제 보충")
#             # df2에서 무작위로 샘플링하여 리스트에 추가
#             sampled_data = candidates.sample(n=num_to_sample, random_state=42)
#             data_to_add.append(sampled_data)

# # 원본 df1과 보충용 데이터를 합쳐 '보강된' 데이터프레임 생성
# if data_to_add:
#     augmented_df = pd.concat([temp_df1] + data_to_add, ignore_index=True)
# else:
#     augmented_df = temp_df1.copy()

In [ ]:
# print("\n--- 보충 후 분포 (균등화 전) ---")
# print(augmented_df['inhibition_bin'].value_counts().sort_index())
# print("\n")

In [ ]:
# # --- 3단계: 최종 데이터셋을 균등하게 다운샘플링 ---
# print("--- 3. 최종 데이터셋 균등화 (다운샘플링) ---")
# # 보강된 데이터셋에서 가장 적은 구간의 데이터 수를 기준으로 삼음
# final_bin_counts = augmented_df['inhibition_bin'].value_counts()
# if not final_bin_counts.empty:
#     final_sample_size = final_bin_counts.min()
#     print(f"최종 샘플링 크기 (보강된 데이터셋의 최소 구간 개수): {final_sample_size}\n")

#     # group by를 이용해 각 구간별로 final_sample_size 만큼 샘플링
#     # dropna()를 통해 inhibition_bin이 없는 행은 제외
#     train = augmented_df.dropna(subset=['inhibition_bin']).groupby('inhibition_bin', group_keys=False).apply(
#         lambda x: x.sample(n=final_sample_size, random_state=42)
#     )
# else:
#     train = pd.DataFrame(columns=augmented_df.columns) # 빈 데이터프레임

# # 임시로 사용한 'inhibition_bin' 열 제거
# train = train.drop(columns=['inhibition_bin'])
# temp_df1 = temp_df1.drop(columns=['inhibition_bin'])
# temp_df2 = temp_df2.drop(columns=['inhibition_bin'])

In [ ]:
# # --- 최종 결과 확인 ---
# print("--- 최종 결과 ---")
# print(f"원본 temp_df1 크기: {temp_df1.shape}")
# print(f"보충용 temp_df2 크기: {temp_df2.shape}")
# print(f"최종 훈련 데이터(train) 크기: {train.shape}")

# # 최종 데이터셋의 분포 확인
# train['inhibition_bin'] = pd.cut(train['Inhibition'], bins=bins, right=False, labels=labels)
# print("\n최종 train 데이터셋의 분포:")
# print(train['inhibition_bin'].value_counts().sort_index())

In [ ]:
print(f"훈련 데이터 : {train.shape}")
print(f"테스트 데이터 : {test.shape}")

In [ ]:
def get_molecule_descriptors(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return [0] * 18, [0] * 2048

        basic_descriptors = [
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumRotatableBonds(mol),
            Descriptors.NumAromaticRings(mol),
            Descriptors.NumHeteroatoms(mol),
            Descriptors.FractionCSP3(mol),
            Descriptors.NumAliphaticRings(mol),
            Lipinski.NumAromaticHeterocycles(mol),
            Lipinski.NumSaturatedHeterocycles(mol),
            Lipinski.NumAliphaticHeterocycles(mol),
            Descriptors.HeavyAtomCount(mol),
            Descriptors.RingCount(mol),
            Descriptors.NOCount(mol),
            Descriptors.NHOHCount(mol),
            Descriptors.NumRadicalElectrons(mol),
        ]

        morgan_fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048, useFeatures=True, useChirality=True)
        morgan_features = [int(bit) for bit in morgan_fp.ToBitString()]

        return basic_descriptors, morgan_features
    except:
        return [0] * 18, [0] * 2048

In [ ]:

result_series = train['Canonical_Smiles'].apply(get_molecule_descriptors)
train['des'], train['ecfp'] = list(zip(*result_series))
# train['des'], train['ecfp'] = train['Canonical_Smiles'].apply(get_molecule_descriptors)

train_des_list =train['des'].tolist()
train_ecfp_list =train['ecfp'].tolist()

feature_lengths_des = [len(x) for x in train_des_list]

if len(set(feature_lengths_des)) != 1:
    max_length = max(feature_lengths_des)
    train_des_list = [x + [0] * (max_length - len(x)) for x in train_des_list]

feature_lengths_ecfp = [len(x) for x in train_ecfp_list]

if len(set(feature_lengths_ecfp)) != 1:
    max_length = max(feature_lengths_ecfp)
    train_ecfp_list = [x + [0] * (max_length - len(x)) for x in train_ecfp_list]

y_train = train['Inhibition'].values

result_series_test = test['Canonical_Smiles'].apply(get_molecule_descriptors)
test['des'], test['ecfp'] = list(zip(*result_series_test))
# test['des'], test['ecfp'] = test['Canonical_Smiles'].apply(get_molecule_descriptors)

test_des_list =test['des'].tolist()
test_ecfp_list =test['ecfp'].tolist()

feature_lengths_des_test = [len(x) for x in test_des_list]

if len(set(feature_lengths_des_test)) != 1:
    max_length = max(feature_lengths_des_test)
    train_des_list = [x + [0] * (max_length - len(x)) for x in test_des_list]

feature_lengths_ecfp_test = [len(x) for x in test_ecfp_list]

if len(set(feature_lengths_ecfp_test)) != 1:
    max_length = max(feature_lengths_ecfp_test)
    train_ecfp_list = [x + [0] * (max_length - len(x)) for x in test_ecfp_list]

In [ ]:
# print("분자 특성 추출 중...")
# train['features'] = train['Canonical_Smiles'].apply(get_molecule_descriptors)

# X_train_list = train['features'].tolist()
# feature_lengths = [len(x) for x in X_train_list]

# if len(set(feature_lengths)) != 1:
#     max_length = max(feature_lengths)
#     X_train_list = [x + [0] * (max_length - len(x)) for x in X_train_list]

# X_train = np.array(X_train_list)
# y_train = train['Inhibition'].values

# test['features'] = test['Canonical_Smiles'].apply(get_molecule_descriptors)
# X_test_list = test['features'].tolist()
# feature_lengths = [len(x) for x in X_test_list]

# if len(set(feature_lengths)) != 1:
#     max_length = max(feature_lengths)
#     X_test_list = [x + [0] * (max_length - len(x)) for x in X_test_list]

# if X_train.shape[1] != len(X_test_list[0]):
#     diff = abs(X_train.shape[1] - len(X_test_list[0]))
#     if X_train.shape[1] > len(X_test_list[0]):
#         X_test_list = [x + [0] * diff for x in X_test_list]
#     else:
#         X_train = np.array([x.tolist() + [0] * diff for x in X_train])

# X_test = np.array(X_test_list)

# print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

scaler = StandardScaler()
X_train_scaled_des = scaler.fit_transform(np.array(train_des_list))
X_train_scaled_ecfp = scaler.fit_transform(np.array(train_ecfp_list))

X_test_scaled_des = scaler.fit_transform(np.array(test_des_list))
X_test_scaled_ecfp = scaler.fit_transform(np.array(test_ecfp_list))

# print(f"전체 훈련 데이터 형태: {X_train_scaled.shape}")
print(f"Autoencoder 입력 차원 (특성의 개수): {X_train_scaled_ecfp.shape[1]}")

In [ ]:
# -- 하이퍼파라미터 설정 --
# 원본 특징의 차원 (StandardScaler를 거친 후의 차원)
input_dim = X_train_scaled_ecfp.shape[1]
# 압축하여 표현할 잠재 공간의 차원 (이 값을 조절하여 압축률 결정)
latent_dim = 256  # 예시: 64, 128, 256 등으로 설정 가능
epochs = 230
batch_size = 32
learning_rate = 5e-5

# -- PyTorch가 GPU를 사용할 수 있는지 확인 --
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Autoencoder 모델 클래스 정의
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()
        # 인코더: 입력 차원을 잠재 공간 차원으로 압축
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.LeakyReLU(),
            nn.Linear(1024, 512),
            nn.LeakyReLU(),
            nn.Linear(512, latent_dim)
        )
        # 디코더: 잠재 공간 벡터를 원본 차원으로 복원
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 1024),
            nn.LeakyReLU(),
            nn.Linear(1024, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# 2. 데이터 준비 (Numpy -> PyTorch Tensor -> DataLoader)
# Autoencoder는 레이블(y) 없이 자기 자신을 복원하도록 학습하므로 X_train_scaled만 사용
train_tensor = torch.FloatTensor(X_train_scaled_ecfp)
train_dataset = TensorDataset(train_tensor, train_tensor) # 입력과 타겟이 동일
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

# 3. 모델 훈련
model = Autoencoder(input_dim, latent_dim).to(device)
criterion = nn.MSELoss()  # 복원 오차를 측정하기 위한 손실 함수 (Mean Squared Error)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("Start training Autoencoder...")
for epoch in range(epochs):
    for data in train_loader:
        inputs, _ = data
        inputs = inputs.to(device)
        
        # 순전파
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        
        # 역전파
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

print("Finished training Autoencoder.")

# 4. 훈련된 인코더를 사용하여 특징 추출 (차원 축소)
model.eval()  # 모델을 평가 모드로 설정
with torch.no_grad(): # 그래디언트 계산 비활성화
    # 훈련 데이터를 저차원 벡터로 변환
    X_train_tensor_scaled = torch.FloatTensor(X_train_scaled_ecfp).to(device)
    X_train_latent_tensor = model.encoder(X_train_tensor_scaled)
    X_train_latent = X_train_latent_tensor.cpu().numpy()

    # 테스트 데이터를 저차원 벡터로 변환
    X_test_tensor_scaled = torch.FloatTensor(X_test_scaled_ecfp).to(device)
    X_test_latent_tensor = model.encoder(X_test_tensor_scaled)
    X_test_latent = X_test_latent_tensor.cpu().numpy()

# print(f"Original X_train shape: {X_train_scaled.shape}")
# print(f"Latent X_train shape: {X_train_latent.shape}")
# print(f"Original X_test shape: {X_test_scaled.shape}")
# print(f"Latent X_test shape: {X_test_latent.shape}")


# <<<           PyTorch Autoencoder 코드 종료                 >>> #
# <<< ======================================================= >>> #

X_train_scaled = np.concatenate((X_train_latent, X_train_scaled_des), axis = 1)
X_test_scaled = np.concatenate((X_test_latent, X_test_scaled_des), axis = 1)


# X_train_scaled = X_train_latent
# X_test_scaled = X_test_latent

In [ ]:
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.1, random_state=42
)

In [ ]:

def normalized_rmse(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return rmse / (np.max(y_true) - np.min(y_true))

def pearson_correlation(y_true, y_pred):
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    return np.clip(corr, 0, 1)

def competition_score(y_true, y_pred):
    nrmse = min(normalized_rmse(y_true, y_pred), 1)
    pearson = pearson_correlation(y_true, y_pred)
    return 0.5 * (1 - nrmse) + 0.5 * pearson

def train_and_evaluate_model(model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    val_nrmse = normalized_rmse(y_val, y_val_pred)
    val_pearson = pearson_correlation(y_val, y_val_pred)
    val_score = competition_score(y_val, y_val_pred)
    print(f"검증 NRMSE: {val_nrmse:.4f}")
    print(f"검증 Pearson: {val_pearson:.4f}")
    print(f"검증 점수: {val_score:.4f}")
    return model, val_score

In [ ]:

# models = {
#     "XGBoost": xgb.XGBRegressor(
#         n_estimators=500, learning_rate=0.05, max_depth=6,
#         subsample=0.8, colsample_bytree=0.8, gamma=0,
#         reg_alpha=0.1, reg_lambda=1, random_state=42,
#         verbosity=0  # XGBoost 경고 제거
#     ),
#     "LightGBM": lgb.LGBMRegressor(
#         n_estimators=500, learning_rate=0.05, num_leaves=31,
#         max_depth=6, subsample=0.8, colsample_bytree=0.8,
#         reg_alpha=0.1, reg_lambda=1, random_state=42,
#         verbosity=-1  # LightGBM 경고 제거
#     ),
#     "GradientBoosting": GradientBoostingRegressor(
#         n_estimators=300, learning_rate=0.05, max_depth=5,
#         min_samples_split=5, min_samples_leaf=2, subsample=0.8,
#         random_state=42
#     ),
#     "RandomForest": RandomForestRegressor(
#         n_estimators=300, max_depth=10, min_samples_split=5,
#         min_samples_leaf=2, random_state=42
#     )
# }

# best_score = -np.inf
# best_model_name = None
# trained_models = {}

# for name, model in models.items():
#     print(f"\n{name} 모델 학습 중...")
#     trained_model, val_score = train_and_evaluate_model(
#         model, X_train_final, y_train_final, X_val, y_val
#     )
#     trained_models[name] = trained_model
#     if val_score > best_score:
#         best_score = val_score
#         best_model_name = name

# print(f"\n최고 성능 모델: {best_model_name}, 검증 점수: {best_score:.4f}")

In [ ]:
# from sklearn.linear_model import ElasticNet

# models = {
#     "ElasticNet": ElasticNet(
#         alpha=5, 
#         l1_ratio=0.5, 
#         random_state=42)
# }

# trained_model, val_score = train_and_evaluate_model(
#     model, X_train_final, y_train_final, X_val, y_val
# )

# best_score = -np.inf
# best_model_name = None
# trained_models = {}

# for name, model in models.items():
#     print(f"\n{name} 모델 학습 중...")
#     trained_model, val_score = train_and_evaluate_model(
#         model, X_train_final, y_train_final, X_val, y_val
#     )
#     trained_models[name] = trained_model
#     if val_score > best_score:
#         best_score = val_score
#         best_model_name = name

# print(f"\n학습 완료 모델: {best_model_name}, 검증 점수: {best_score:.4f}")

In [ ]:
from sklearn.metrics import  make_scorer
competition_scorer = make_scorer(competition_score, greater_is_better=True)

from sklearn.model_selection import KFold, cross_val_score

# K-Fold 교차 검증 설정
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# XGBoost 모델
xgb_model = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, gamma=0,
        reg_alpha=0.1, reg_lambda=1, random_state=42,
        verbosity=0  # XGBoost 경고 제거
    )
# 교차 검증 수행 시, scoring 인자에 만들어둔 scorer를 전달
scores = cross_val_score(xgb_model, X_train_final, y_train_final, cv=kf, scoring=competition_scorer)

print(f"XGBoost 교차 검증 점수 (Competition Score): {scores}")
print(f"평균 점수: {np.mean(scores):.4f}, 표준편차: {np.std(scores):.4f}")

In [ ]:

# final_model = models[best_model_name]
final_model = xgb_model
final_model.fit(X_train_scaled, y_train)
test_preds = final_model.predict(X_test_scaled)

submission['Inhibition'] = test_preds
submission.to_csv('improved_submission1234.csv', index=False)
print("예측 결과 저장: improved_submission.csv")

In [ ]:
plt.figure(figsize=(10, 6))
# y_val_pred = trained_models[best_model_name].predict(X_val)
y_val_pred = final_model.predict(X_val)
plt.scatter(y_val, y_val_pred, alpha=0.5)
plt.plot([min(y_val), max(y_val)], [min(y_val), max(y_val)], 'r--')
plt.xlabel('real')
plt.ylabel('predict')
plt.title(f'xgb predict')
plt.savefig('model_performance.png')
print("모델 성능 시각화 저장: model_performance.png")

if hasattr(final_model, 'feature_importances_'):
    n_features = 20
    importances = final_model.feature_importances_
    indices = np.argsort(importances)[::-1][:n_features]
    plt.figure(figsize=(12, 8))
    plt.title('important feature')
    plt.bar(range(n_features), importances[indices])
    plt.xticks(range(n_features), indices, rotation=90)
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    print("특성 중요도 시각화 저장: feature_importance.png")